## Scrapeo de Documentos 23-F

In [1]:
import requests
from lxml import html
from tqdm import tqdm
import pandas as pd
import os
import hashlib

BASE = "https://23fbuscador.rtve.es"
headers = {"User-Agent": "Mozilla/5.0"}


# -----------------------------
# Helper: Get total number of pages
# -----------------------------
def get_total_pages():
    r = requests.get(f"{BASE}/?page_size=200&page=1", headers=headers)
    tree = html.fromstring(r.content)

    text = tree.xpath("//span[@class='nav-position']/text()")
    if not text:
        return 1

    # Example: "Página 1 de 7"
    parts = text[0].split()
    return int(parts[-1])


# -----------------------------
# Helper: Extract document rows from a listing page
# -----------------------------
def get_page_documents(page_number):
    url = f"{BASE}/?page_size=200&page={page_number}"
    r = requests.get(url, headers=headers)
    tree = html.fromstring(r.content)

    rows = tree.xpath("//table/tbody/tr")

    docs = []
    for row in rows:
        title = row.xpath(".//td[1]/a/text()")
        href = row.xpath(".//td[1]/a/@href")
        pages = row.xpath(".//td[2]/text()")
        summary = row.xpath(".//td[4]/text()")
        tags = row.xpath(".//td[5]//span[@class='tag-chip']/text()")

        if not href:
            continue

        docs.append({
            "title": title[0].strip() if title else "",
            "url": BASE + href[0],
            "pages": pages[0].strip() if pages else None,
            "summary": summary[0].strip() if summary else "",
            "tags": tags,
        })

    return docs


# -----------------------------
# Helper: Download transcript text from a document page
# -----------------------------
def download_transcript(url):
    r = requests.get(url, headers=headers)
    tree = html.fromstring(r.content)

    pre = tree.xpath("//pre/text()")
    if not pre:
        return None

    return pre[0]


# -----------------------------
# Helper: Generate safe filename
# -----------------------------
def make_safe_filename(title):
    # Remove illegal characters
    cleaned = "".join(c for c in title if c.isalnum() or c in " _-").strip()

    # If too long, shorten and add a hash
    if len(cleaned) > 100:
        hash_suffix = hashlib.md5(cleaned.encode()).hexdigest()[:8]
        cleaned = cleaned[:80] + "_" + hash_suffix

    return cleaned + ".txt"


# -----------------------------
# Main scraper
# -----------------------------
def scrape_all():
    os.makedirs("transcripts", exist_ok=True)

    total_pages = get_total_pages()
    print(f"Total pages: {total_pages}")

    all_docs = []

    # Step 1: Collect metadata for all documents
    for page in range(1, total_pages + 1):
        docs = get_page_documents(page)
        all_docs.extend(docs)

    print(f"Found {len(all_docs)} documents")

    # Step 2: Download transcripts + save .txt files
    for doc in tqdm(all_docs):
        transcript = download_transcript(doc["url"])
        doc["transcript"] = transcript

        if transcript:
            filename = make_safe_filename(doc["title"])
            filepath = os.path.join("transcripts", filename)

            with open(filepath, "w", encoding="utf-8") as f:
                f.write(transcript)

            doc["filename"] = filename
        else:
            doc["filename"] = None

    # Step 3: Convert to DataFrame
    df = pd.DataFrame(all_docs)
    df.to_csv("23f_scrappedDF.csv", index=False, encoding="utf-8")
    return df

In [2]:
# Run scraper
df = scrape_all()

Total pages: 1
Found 167 documents


100%|██████████| 167/167 [01:30<00:00,  1.85it/s]


In [3]:
df.head(5)

,title,url,pages,summary,tags,transcript,filename
0,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1860?...,3,El juicio oral 2/81 celebrado en febrero de 19...,"[C/SG/2820/20-02-82, DTOR., Vista oral 2/81]",C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\...,Vista oral 281 del Consejo Supremo de Justicia...
1,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1859?...,4,Resumen global del documento:\n\nEl documento ...,"[C/SG/2896/22-02-82, Vista oral 2/81, Consejo ...",C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nAS...,Vista oral 281 del Consejo Supremo de Justicia...
2,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1858?...,5,Resumen global del documento:\n\nEl documento ...,"[C/SG/2992/24-02-82, Vista Oral 2/81, Consejo ...",C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nAS...,Vista oral 281 del Consejo Supremo de Justicia...
3,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1857?...,6,El documento recoge el desarrollo de la sesión...,"[C/SG/3.081/25-02-82, Vista Oral 2/81, Consejo...",C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nA...,Vista oral 281 del Consejo Supremo de Justicia...
4,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1856?...,6,Resumen global del documento sobre la sesión d...,"[C/SG/3.249/26-02-82, SG, Consejo Supremo de J...",C/SG/3.249/26-02-82\nSG\n\n# NOTA INFORMATIVA\...,Vista oral 281 del Consejo Supremo de Justicia...


In [9]:
from tabulate import tabulate
print(tabulate(df.head(1), headers='keys', tablefmt='psql'))

+----+----------------------------------------------------------------------------------+--------------------------------------------------------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------+
|    | 


## Carga de Archivo Scrapeado



In [ ]:
from google.colab import files
files.upload()

Saving 23f_scrappedDF.csv to 23f_scrappedDF (1).csv


{'23f_scrappedDF (1).csv': b'title,url,pages,summary,tags,transcript,filename\nVista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,https://23fbuscador.rtve.es/document/ocr/1860?page_size=200&page=1,3,"El juicio oral 2/81 celebrado en febrero de 1982 se caracteriz\xc3\xb3 por un intenso desarrollo en sus primeras sesiones, con declaraciones parciales de altos mandos militares y certificaciones oficiales, aunque plagado de controversias por la interpretaci\xc3\xb3n y selecci\xc3\xb3n de testimonios, especialmente en re...","[\'C/SG/2820/20-02-82\', \'DTOR.\', \'Vista oral 2/81\']","C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SESI\xc3\x93N CORRESPONDIENTE AL 20-02-82\n\n- Solo ha tenido lugar la sesi\xc3\xb3n de la ma\xc3\xb1ana. Empez\xc3\xb3 a las 10,06 horas.\n\n- Durante la sesi\xc3\xb3n han tenido lugar, a petici\xc3\xb3n del Sr. Fiscal las declaraciones siguientes:\n\n. Parcial de Teniente Coronel D. L

In [ ]:
df = pd.read_csv('23f_scrappedDF.csv')

In [ ]:
df.head()

,title,url,pages,summary,tags,transcript,filename
0,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1860?...,3,El juicio oral 2/81 celebrado en febrero de 19...,"['C/SG/2820/20-02-82', 'DTOR.', 'Vista oral 2/...",C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\...,Vista oral 281 del Consejo Supremo de Justicia...
1,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1859?...,4,Resumen global del documento:\n\nEl documento ...,"['C/SG/2896/22-02-82', 'Vista oral 2/81', 'Con...",C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nAS...,Vista oral 281 del Consejo Supremo de Justicia...
2,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1858?...,5,Resumen global del documento:\n\nEl documento ...,"['C/SG/2992/24-02-82', 'Vista Oral 2/81', 'Con...",C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nAS...,Vista oral 281 del Consejo Supremo de Justicia...
3,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1857?...,6,El documento recoge el desarrollo de la sesión...,"['C/SG/3.081/25-02-82', 'Vista Oral 2/81', 'Co...",C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nA...,Vista oral 281 del Consejo Supremo de Justicia...
4,Vista oral 2/81 del Consejo Supremo de Justici...,https://23fbuscador.rtve.es/document/ocr/1856?...,6,Resumen global del documento sobre la sesión d...,"['C/SG/3.249/26-02-82', 'SG', 'Consejo Supremo...",C/SG/3.249/26-02-82\nSG\n\n# NOTA INFORMATIVA\...,Vista oral 281 del Consejo Supremo de Justicia...


In [ ]:
df.columns

Index(['title', 'url', 'pages', 'summary', 'tags', 'transcript', 'filename'], dtype='object')

In [ ]:
print(df['transcript'][0])

C/SG/2820/20-02-82
DTOR.

NOTA INFORMATIVA

ASUNTO: Vista oral 2/81

1.- DESARROLLO DE LA SESIÓN CORRESPONDIENTE AL 20-02-82

- Solo ha tenido lugar la sesión de la mañana. Empezó a las 10,06 horas.

- Durante la sesión han tenido lugar, a petición del Sr. Fiscal las declaraciones siguientes:

. Parcial de Teniente Coronel D. Luis Arana Lorite (Ayte. Gral. Lluch).

. Parcial de Teniente Coronel D. Manuel Miler Hidalgo (Ayte. Gral. Esquivias).

. 1ª, 2ª, 3ª y 4ª del Teniente Coronel TEJERO.

. Careo Teniente Coronel TEJERO - SR. CARRES.

- Descanso de 11,50 horas a 12,13 horas.

. Careo Teniente Coronel TEJERO - Capitán GOMEZ IGLESIAS

. Certificación del Presidente del Congreso Sr. LAVILLA.

. Certificación sobre la situación militar del TCOL.TEJERO.

. 1ª y 3ª del Coronel D. DIEGO IBAÑEZ INGLES

. Certificación del entonces Gobernador Civil de Valencia D.José María Fernández del Rio Fernández

. 1ª y 3ª del Teniente Coronel MAS OLIVER.

La vista se interrumpió hasta las 10,00 horas de